In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from pydantic import BaseModel, Field
import os
from mistralai import Mistral
from mistralai.models.chatcompletionrequest import Messages
from mistralai.models.systemmessage import SystemMessage
from mistralai.models.usermessage import UserMessage
from mistralai.models.imageurlchunk import ImageURLChunk

api_key = os.environ["MISTRAL_API_KEY"]

client = Mistral(api_key=api_key)
model = "mistral-medium-2508"


class SingleExpense(BaseModel):
    title: str = Field(...)
    description: str | None
    amount: int = Field(
        ..., description="Amount in NPR. Round it to the ceiling if in decimal."
    )
    transaction_date: str | None = Field(
        ..., description="Transaction date in yyyy-mm-dd format, if available"
    )


class Expenses(BaseModel):
    expenses: list[SingleExpense]


system_prompt: str = """
You are a financial recept parser. You will be given a bill and your job is to parse 
the contents of the bill and create a list of itemized items.
"""

image_url = "https://media.discordapp.net/attachments/1366030884767141988/1433098013139664906/rn_image_picker_lib_temp_e1db0a79-71a2-44e7-a9a6-ab6b75f689b7.jpg?ex=690ffa48&is=690ea8c8&hm=84bb3feb3b3fcc2f59046850ddf4eddfa694c99810a21473d2aa54a8f24aa4bd&=&format=webp&width=1308&height=1744"
image_url = "https://media.discordapp.net/attachments/1366030884767141988/1436724247484563507/rn_image_picker_lib_temp_81868b55-6099-491f-9aa0-3af19b822809.jpg?ex=6910a539&is=690f53b9&hm=ea996cca38c9b02dede09e9903a0c87865f1eaa6fd975b582c533ebac855812e&=&format=webp&width=1308&height=1744"

messages: list[Messages] = [
    SystemMessage(role="system", content=system_prompt),
    UserMessage(
        content=[
            ImageURLChunk(
                image_url=image_url,
                type="image_url",
            )
        ]
    ),
]

chat_response = client.chat.parse(
    model=model,
    messages=messages,
    response_format=Expenses,
    temperature=0,
)

In [17]:
chat_response.model_dump()

{'id': '4208d7189ac94ed2b82e6003601cea0a',
 'object': 'chat.completion',
 'model': 'mistral-medium-2508',
 'usage': {'prompt_tokens': 2406,
  'completion_tokens': 124,
  'total_tokens': 2530},
 'created': 1762612223,
 'choices': [{'index': 0,
   'message': {'content': '{\n  "expenses": [\n    {\n      "title": "POKEMON TRA HSC",\n      "description": "POKEMON TRA HSC",\n      "amount": 305,\n      "transaction_date": "2082-07-20"\n    },\n    {\n      "title": "MAGNETIC BU HSC",\n      "description": "MAGNETIC BU HSC",\n      "amount": 1550,\n      "transaction_date": "2082-07-20"\n    }\n  ]\n}',
    'tool_calls': None,
    'prefix': False,
    'role': 'assistant',
    'parsed': {'expenses': [{'title': 'POKEMON TRA HSC',
       'description': 'POKEMON TRA HSC',
       'amount': 305,
       'transaction_date': '2082-07-20'},
      {'title': 'MAGNETIC BU HSC',
       'description': 'MAGNETIC BU HSC',
       'amount': 1550,
       'transaction_date': '2082-07-20'}]}},
   'finish_reason':